# Clase 3 — Tu primer clasificador de sentimientos

**Objetivos de hoy**
- Transformar texto en números con una bolsa de palabras.
- Entrenar un clasificador que distinga reseñas positivas de negativas.
- Comprobar el efecto de tener más datos de entrenamiento.

Consulta el [capítulo 2 del manual](../manual/02_sentiment_analysis.md) para la explicación completa.

## 1. Carga el corpus de reseñas de cine

Usamos `pos.txt` y `neg.txt`, incluidos en el material del curso: más de cinco mil frases positivas y cinco mil negativas.

In [ ]:
import tensorflow as tf

with open("../pos.txt", encoding="latin-1") as f:
    positivas = [linea.strip() for linea in f if linea.strip()]
with open("../neg.txt", encoding="latin-1") as f:
    negativas = [linea.strip() for linea in f if linea.strip()]

textos = positivas + negativas
etiquetas = [1] * len(positivas) + [0] * len(negativas)

print("Frases positivas:", len(positivas))
print("Frases negativas:", len(negativas))
print("Ejemplo positivo:", positivas[0])
print("Ejemplo negativo:", negativas[0])

## 2. Divide en entrenamiento y test, y mezcla los datos

In [ ]:
import numpy as np

indices = np.arange(len(textos))
np.random.seed(42)
np.random.shuffle(indices)

textos = np.array(textos)[indices]
etiquetas = np.array(etiquetas)[indices]

corte = int(len(textos) * 0.8)
textos_entreno, textos_test = textos[:corte], textos[corte:]
etiquetas_entreno, etiquetas_test = etiquetas[:corte], etiquetas[corte:]

print("Frases de entrenamiento:", len(textos_entreno))
print("Frases de test:", len(textos_test))

## 3. Construye el vocabulario con `TextVectorization`

Esta capa cuenta las palabras más frecuentes en tus datos y las convierte en una bolsa de palabras.

In [ ]:
vectorizador = tf.keras.layers.TextVectorization(max_tokens=5000, output_mode="multi_hot")
vectorizador.adapt(textos_entreno)

print("Tamaño del vocabulario:", len(vectorizador.get_vocabulary()))
print("Primeras 10 palabras:", vectorizador.get_vocabulary()[:10])

## 4. Define y entrena el clasificador

In [ ]:
modelo = tf.keras.Sequential([
    vectorizador,
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(textos_entreno, etiquetas_entreno, epochs=10, validation_split=0.1)

## 5. Evalúa y prueba con frases nuevas

In [ ]:
perdida_test, exactitud_test = modelo.evaluate(textos_test, etiquetas_test)
print(f"Exactitud sobre el conjunto de test: {exactitud_test:.2%}")

frases_nuevas = [
    "una actuación brillante y una historia conmovedora",
    "el peor guion que he visto en años",
]
print(modelo.predict(frases_nuevas))

## Resumen

- Convertiste texto en vectores numéricos con una bolsa de palabras, sin escribir el vocabulario a mano.
- Entrenaste un clasificador binario (positivo/negativo) con una sola neurona de salida y activación sigmoide.
- El material original de esta lección muestra cómo el mismo pipeline, aplicado sobre Sentiment140 (un millón seiscientos mil tuits), mejora la exactitud gracias a tener muchos más datos.

Continúa con el [notebook de práctica de esta clase](../notebooks_practica/clase03_practica_sentimiento.ipynb).